In [1]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from tqdm import tqdm
from pathlib import Path
from data_loader import build_complete_dataset
from window import create_record_windows


In [2]:
def extract_window_features(window_df, prefix=""):
    features = {}
    columns = [column for column in window_df.columns if column != "Time" ]
    data = window_df.drop(columns=["Time"], errors="ignore").values
    for i, col in enumerate(columns):
            signal = data[:, i]
            features[f"{prefix}_{col}_mean"] = np.mean(signal)
            features[f"{prefix}_{col}_std"] = np.std(signal)
            features[f"{prefix}_{col}_rms"] = np.sqrt(np.mean(signal**2))
            features[f"{prefix}_{col}_max"] = np.max(signal)
            features[f"{prefix}_{col}_min"] = np.min(signal)
            features[f"{prefix}_{col}_ptp"] = np.ptp(signal)
            features[f"{prefix}_{col}_skew"] = skew(signal) if len(signal) > 3 else 0
            features[f"{prefix}_{col}_kurtosis"] = kurtosis(signal) if len(signal) > 3 else 0
            fft_vals = np.abs(np.fft.rfft(signal))
            features[f"{prefix}_{col}_fft_energy"] = np.sum(fft_vals**2)
            features[f"{prefix}_{col}_fft_peak_freq"] = np.argmax(fft_vals)
    return features

In [3]:
def process_record_features(record, create_record_windows_func):
    acc_windows, gyro_windows, mic_windows = create_record_windows_func(record)
    
    record_features = []
    min_len = min(len(acc_windows), len(gyro_windows), len(mic_windows))
    
    for i in range(min_len):
        features = {}
        features.update(extract_window_features(acc_windows[i], prefix="acc"))
        features.update(extract_window_features(gyro_windows[i], prefix="gyro"))
        features.update(extract_window_features(mic_windows[i], prefix="mic"))
        
        features["segment_id"] = record.metadata.get("segment_id")
        features["split_label"] = record.metadata.get("split_label")
        features["anomaly_label"] = record.metadata.get("anomaly_label")
        features["domain_shift_op"] = record.metadata.get("domain_shift_op")
        features["domain_shift_env"] = record.metadata.get("domain_shift_env")
        record_features.append(features)
        
    return pd.DataFrame(record_features)

In [4]:
def build_feature_dataset(dataset, create_record_windows_func):
    results = []
    for record in tqdm(dataset, desc="Ekstrakcja cech"):
        df_feats = process_record_features(record, create_record_windows_func)
        results.append(df_feats)
    return pd.concat(results, ignore_index=True)

In [18]:
path = Path("../data")
train_path = path / "X_train.csv"
test_path = path / "X_test.csv"

if train_path.exists() and test_path.exists():
    X_train_df = pd.read_csv(train_path)
    X_test_df = pd.read_csv(test_path)
else:
    df = build_complete_dataset()
    X_train_df = build_feature_dataset(df[0], create_record_windows)
    X_test_df = build_feature_dataset(df[1], create_record_windows)
    X_train_df.to_csv(train_path, index=False)
    X_test_df.to_csv(test_path, index=False)

print("Train:", X_train_df.shape)
print("Test :", X_test_df.shape)

print(X_train_df.columns.tolist())
print(X_train_df.columns.tolist())


Train: (102984, 75)
Test : (25984, 75)
['acc_A_x [g]_mean', 'acc_A_x [g]_std', 'acc_A_x [g]_rms', 'acc_A_x [g]_max', 'acc_A_x [g]_min', 'acc_A_x [g]_ptp', 'acc_A_x [g]_skew', 'acc_A_x [g]_kurtosis', 'acc_A_x [g]_fft_energy', 'acc_A_x [g]_fft_peak_freq', 'acc_A_y [g]_mean', 'acc_A_y [g]_std', 'acc_A_y [g]_rms', 'acc_A_y [g]_max', 'acc_A_y [g]_min', 'acc_A_y [g]_ptp', 'acc_A_y [g]_skew', 'acc_A_y [g]_kurtosis', 'acc_A_y [g]_fft_energy', 'acc_A_y [g]_fft_peak_freq', 'acc_A_z [g]_mean', 'acc_A_z [g]_std', 'acc_A_z [g]_rms', 'acc_A_z [g]_max', 'acc_A_z [g]_min', 'acc_A_z [g]_ptp', 'acc_A_z [g]_skew', 'acc_A_z [g]_kurtosis', 'acc_A_z [g]_fft_energy', 'acc_A_z [g]_fft_peak_freq', 'gyro_G_x [mdps]_mean', 'gyro_G_x [mdps]_std', 'gyro_G_x [mdps]_rms', 'gyro_G_x [mdps]_max', 'gyro_G_x [mdps]_min', 'gyro_G_x [mdps]_ptp', 'gyro_G_x [mdps]_skew', 'gyro_G_x [mdps]_kurtosis', 'gyro_G_x [mdps]_fft_energy', 'gyro_G_x [mdps]_fft_peak_freq', 'gyro_G_y [mdps]_mean', 'gyro_G_y [mdps]_std', 'gyro_G_y [mdps]_

In [19]:
X_train_df_source = X_train_df[X_train_df["split_label"] == "Normal_Source_Train"].copy()
X_test_df_source = X_test_df[X_test_df["split_label"].isin(["Normal_Source_Test", "Anomaly_Source_Test"])].copy()

X_train_df_target = X_train_df[X_train_df["split_label"] == "Normal_Target_Train"].copy()
X_test_df_target = X_test_df[X_test_df["split_label"].isin(["Normal_Target_Test", "Anomaly_Target_Test"])].copy()

In [20]:
metadata_cols = ["segment_id", "anomaly_label", "domain_shift_op", "domain_shift_env"]

acc_features = [col for col in X_train_df.columns if col.startswith("acc_")]
gyro_features = [col for col in X_train_df.columns if col.startswith("gyro_")]
mic_features = [col for col in X_train_df.columns if col.startswith("mic_")]

In [21]:
X_train_acc = X_train_df[acc_features].fillna(0)
X_test_acc = X_test_df[acc_features].fillna(0)

X_train_mic = X_train_df[mic_features].fillna(0)
X_test_mic = X_test_df[mic_features].fillna(0)

X_train_gyro = X_train_df[gyro_features].fillna(0)
X_test_gyro = X_test_df[gyro_features].fillna(0)

X_train_source_acc = X_train_df_source[acc_features].fillna(0)
X_train_source_mic = X_train_df_source[mic_features].fillna(0)
X_train_source_gyro = X_train_df_source[gyro_features].fillna(0)

X_test_source_acc = X_test_df_source[acc_features].fillna(0)
X_test_source_mic = X_test_df_source[mic_features].fillna(0)
X_test_source_gyro = X_test_df_source[gyro_features].fillna(0)

X_train_target_acc = X_train_df_target[acc_features].fillna(0)
X_train_target_mic = X_train_df_target[mic_features].fillna(0)
X_train_target_gyro = X_train_df_target[gyro_features].fillna(0)

X_test_target_acc = X_test_df_target[acc_features].fillna(0)
X_test_target_mic = X_test_df_target[mic_features].fillna(0)
X_test_target_gyro = X_test_df_target[gyro_features].fillna(0)


In [85]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)


def run_isolation_forest(X_train, X_test, train_df, test_df, sensor_type):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = IsolationForest(
        n_estimators=300,
        contamination='auto',
        max_features=0.75,
        max_samples=512,
        random_state=42,
        n_jobs=-1
    ) # to do: hiperparameter tuning

    model.fit(X_train_scaled)

    scores = -model.score_samples(X_test_scaled)
    train_scores = -model.score_samples(X_train_scaled)
    

    results = pd.DataFrame({"segment_id": test_df["segment_id"].values, "anomaly_label": test_df["anomaly_label"].values, "score": scores})
    segment_scores = (results.groupby("segment_id").agg(score=("score", "max"), anomaly_label=("anomaly_label", "first")).reset_index())

    y_true = (segment_scores["anomaly_label"] == "loosescrewsA").astype(int)

    finite = (pd.Series(train_scores).replace([np.inf, -np.inf], np.nan).dropna().values)
    threshold = np.mean(finite) + 2 * np.std(finite) if finite.size else np.inf
    y_pred = (segment_scores["score"] >= threshold).astype(int)
    
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred,zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_true, segment_scores["score"])
    cm = confusion_matrix(y_true, y_pred)
    
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC AUC  : {roc_auc:.4f}")
    print(cm)

    metrics = {"Sensors": sensor_type,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc,
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1]}
    return metrics

In [86]:
results = []
print("ACC")
result_acc = run_isolation_forest(X_train_acc, X_test_acc, X_train_df, X_test_df, "ACC")
results.append(result_acc)

print("MIC")
result_mic = run_isolation_forest(X_train_mic, X_test_mic, X_train_df, X_test_df, "MIC")
results.append(result_mic)

print("GYRO")
result_gyro = run_isolation_forest(X_train_gyro, X_test_gyro, X_train_df, X_test_df, "GYRO")
results.append(result_gyro)

ACC
Accuracy : 0.5797
Precision: 0.5473
Recall   : 0.9224
F1-score : 0.6870
ROC AUC  : 0.5458
[[ 55 177]
 [ 18 214]]
MIC
Accuracy : 0.5582
Precision: 0.5521
Recall   : 0.6164
F1-score : 0.5825
ROC AUC  : 0.5746
[[116 116]
 [ 89 143]]
GYRO
Accuracy : 0.5043
Precision: 0.5042
Recall   : 0.5216
F1-score : 0.5127
ROC AUC  : 0.5123
[[113 119]
 [111 121]]


In [87]:
# wyniki na samym source
results_source = []
print("ACC - source")
result_acc_source = run_isolation_forest(X_train_source_acc, X_test_source_acc, X_train_df_source, X_test_df_source, "ACC - source")
results_source.append(result_acc_source)

print("MIC - source")
result_mic_source = run_isolation_forest(X_train_source_mic, X_test_source_mic, X_train_df_source, X_test_df_source, "MIC - source")
results_source.append(result_mic_source)

print("GYRO - source")
result_gyro_source = run_isolation_forest(X_train_source_gyro, X_test_source_gyro, X_train_df_source, X_test_df_source, "GYRO - source")
results_source.append(result_gyro_source)

ACC - source
Accuracy : 0.6034
Precision: 0.5741
Recall   : 0.8017
F1-score : 0.6691
ROC AUC  : 0.6416
[[47 69]
 [23 93]]
MIC - source
Accuracy : 0.5776
Precision: 0.5584
Recall   : 0.7414
F1-score : 0.6370
ROC AUC  : 0.5841
[[48 68]
 [30 86]]
GYRO - source
Accuracy : 0.5733
Precision: 0.5955
Recall   : 0.4569
F1-score : 0.5171
ROC AUC  : 0.5843
[[80 36]
 [63 53]]
